In [1]:
!nvidia-smi
!git clone -b rfdetr-baseline-karol https://github.com/orzMik/road-damage-detection.git
%cd road-damage-detection

Sun May 31 15:36:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install "rfdetr[train,loggers]" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
!pip install rfdetr kaggle python-dotenv -q

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'username'
os.environ['KAGGLE_KEY'] = 'KEY'  # PAS l'ancienne 5d3d913... qui est compromise

In [12]:
%%writefile scripts/prepare_coco_split.py
"""Prepare COCO splits for RF-DETR training. Remaps category IDs to start at 1
(RF-DETR/Roboflow convention treats id=0 as background)."""
import json
import random
import shutil
from pathlib import Path

PROJECT_ROOT = Path(__file__).parent.parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
IMAGES_DIR = RAW_DIR / "images"
COCO_JSON = RAW_DIR / "annotations_coco.json"
OUTPUT_DIR = DATA_DIR / "coco_split"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
SEED = 42


def main():
    if not COCO_JSON.exists():
        print(f"ERROR: {COCO_JSON} not found.")
        return

    print(f"Loading {COCO_JSON}")
    with open(COCO_JSON) as f:
        coco = json.load(f)

    print(f"  {len(coco['images'])} images, {len(coco['annotations'])} annotations")

    # Remap category IDs: original {0,1,2} -> new {1,2,3}
    # (RF-DETR treats category_id=0 as background)
    id_remap = {cat["id"]: cat["id"] + 1 for cat in coco["categories"]}
    new_categories = [
        {**cat, "id": id_remap[cat["id"]]} for cat in coco["categories"]
    ]
    new_annotations = [
        {**ann, "category_id": id_remap[ann["category_id"]]}
        for ann in coco["annotations"]
    ]
    print(f"  Categories after remap: {[(c['id'], c['name']) for c in new_categories]}")

    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    images = coco["images"]
    random.seed(SEED)
    shuffled = images.copy()
    random.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    splits = {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train : n_train + n_val],
        "test": shuffled[n_train + n_val :],
    }

    for split_name, split_images in splits.items():
        split_dir = OUTPUT_DIR / split_name
        split_dir.mkdir(parents=True)
        image_ids = {img["id"] for img in split_images}
        split_annotations = [a for a in new_annotations if a["image_id"] in image_ids]

        for img in split_images:
            src = IMAGES_DIR / img["file_name"]
            dst = split_dir / img["file_name"]
            if not src.exists():
                print(f"  WARNING: source image not found: {src}")
                continue
            dst.symlink_to(src.resolve())

        split_coco = {
            "info": coco.get("info", {}),
            "licenses": coco.get("licenses", []),
            "categories": new_categories,
            "images": split_images,
            "annotations": split_annotations,
        }
        with open(split_dir / "_annotations.coco.json", "w") as f:
            json.dump(split_coco, f)

        print(f"  {split_name}: {len(split_images)} images, {len(split_annotations)} annotations")

    print(f"\nDone. Output: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Overwriting scripts/prepare_coco_split.py


In [6]:
!python scripts/download_dataset.py

Target directory prepared at: /content/road-damage-detection/data/raw
Authenticating with Kaggle API...
Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
Extracting dataset ZIP: road-damage-dataset-potholes-cracks-and-manholes.zip...
Detected nested top-level 'data' folder. Moving contents into data/raw...
Download complete. Dataset is ready in: /content/road-damage-detection/data/raw


In [14]:
!python scripts/prepare_coco_split.py

Loading /content/road-damage-detection/data/raw/annotations_coco.json
  2009 images, 4737 annotations
  Categories after remap: [(1, 'pothole'), (2, 'crack'), (3, 'manhole')]
  train: 1406 images, 3250 annotations
  valid: 301 images, 725 annotations
  test: 302 images, 762 annotations

Done. Output: /content/road-damage-detection/data/coco_split


In [ ]:
!python scripts/train_rfdetr_baseline.py

=== Training RF-DETR (Base) ===
Dataset: /content/road-damage-detection/data/coco_split
Output:  /content/road-damage-detection/runs/rfdetr_baseline
Epochs:  15
Batch:   4 (effective 16 with grad accum)
LR:      0.0001

/usr/local/lib/python3.12/dist-packages/deprecate/proxy.py:168: FutureWarning: The `RFDETRBase` was deprecated since v1.7.0. It will be removed in v2.0.0.
  stream(msg)
[2026-05-31 15:43:09] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-base.pth already exists with correct MD5 hash.
[2026-05-31 15:43:11] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-base.pth already exists with correct MD5 hash.
[2026-05-31 15:43:14] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-base.pth already exists with correct MD5 hash.
[2026-05-31 15:43:15] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 3. The detection head will be re-initialized to 3 classes.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU 

In [ ]:
!find runs/ -name "*.pth" 2>/dev/null
!find runs/ -name "*.pt" 2>/dev/null

In [ ]:
!ls data/raw/